In [59]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 
%matplotlib inline
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.preprocessing import RobustScaler,LabelEncoder,OneHotEncoder,PowerTransformer
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from xgboost import XGBRegressor
from scipy.stats import skew,boxcox
from lightgbm import LGBMRegressor

In [6]:
df=pd.read_csv('../sources/housing.csv')

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [16]:
def find_outliner_iqr(df,threshold=1.5):
    outlier_summary={}
    numeric_cols=df.select_dtypes(include=["float64",'int64']).columns
    categorical_cols=df.select_dtypes(include=['object']).columns
    for col in numeric_cols:
        Q1=df[col].quantile(0.25)
        Q3=df[col].quantile(0.75)
        IQR=Q3-Q1
        
        lower_bound=Q1-threshold*IQR
        upper_bound=Q3+threshold*IQR
        
        outliers=df[(df[col]<lower_bound)|(df[col]>upper_bound)]
        outlier_summary[col]={
            'outlier_count':outliers.shape[0],
            'outlier_percentage':100 * outliers.shape[0]/df.shape[0],
            'lower_bound':lower_bound,
            'upper_bound':upper_bound
        }
    return pd.DataFrame(outlier_summary)

In [57]:
def inverse_boxcox(y,lamba_):
    if lamba_==0:
        return np.exp(y)
    else:
        return np.power(y*lamba_+1,1/lamba_)

In [22]:
def remove_outliers_column(df,target_col,threshold=1.5):
    Q1=df[target_col].quantile(0.25)
    Q3=df[target_col].quantile(0.75)
    IQR=Q3-Q1
    
    lower_bound=Q1-threshold*IQR
    upper_bound=Q3+threshold*IQR
    return df[(df[target_col]>=lower_bound) & (df[target_col]<=upper_bound)]

In [23]:
find_outliner_iqr(df)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
outlier_count,0.000,0.00,0.0,1287.000000,1271.000000,1196.000000,1220.000000,681.000000,1071.000000
outlier_percentage,0.000,0.00,0.0,6.235465,6.157946,5.794574,5.910853,3.299419,5.188953
lower_bound,-127.485,28.26,-10.5,-1102.625000,-230.500000,-620.000000,-207.500000,-0.706375,-98087.500000
upper_bound,-112.325,43.38,65.5,5698.375000,1173.500000,3132.000000,1092.500000,8.013025,482412.500000


In [25]:
df_target_clean=remove_outliers_column(df,'median_house_value')

In [26]:
df_target_clean=pd.get_dummies(df_target_clean,columns=['ocean_proximity'],drop_first=True)

In [27]:
df_target_clean['total_bedrooms']=df_target_clean['total_bedrooms'].fillna(df_target_clean['total_bedrooms'].median())
X=df_target_clean.drop('median_house_value',axis=1)
y=df_target_clean['median_house_value']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=15)

In [77]:
latest=XGBRegressor(n_estimators=300,max_depth=6,learning_rate=0.1,colsample_bytree=0.7)

In [78]:
latest.fit(X_train,y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [79]:
y_pred=latest.predict(X_test)
print('mse:',mean_squared_error(y_test,y_pred))
print('msa:',mean_absolute_error(y_test,y_pred))
print('rmse:',np.sqrt(mean_squared_error(y_test,y_pred)))
print('score:',r2_score(y_test,y_pred))

mse: 1710300835.2590632
msa: 28162.712415900187
rmse: 41355.78357689603
score: 0.8149638871706426


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [43]:
df_lgb=df.copy()

In [44]:
df_lgb=pd.get_dummies(df_lgb,columns=['ocean_proximity'],drop_first=True)

In [45]:
df_lgb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   longitude                   20640 non-null  float64
 1   latitude                    20640 non-null  float64
 2   housing_median_age          20640 non-null  float64
 3   total_rooms                 20640 non-null  float64
 4   total_bedrooms              20433 non-null  float64
 5   population                  20640 non-null  float64
 6   households                  20640 non-null  float64
 7   median_income               20640 non-null  float64
 8   median_house_value          20640 non-null  float64
 9   ocean_proximity_INLAND      20640 non-null  bool   
 10  ocean_proximity_ISLAND      20640 non-null  bool   
 11  ocean_proximity_NEAR BAY    20640 non-null  bool   
 12  ocean_proximity_NEAR OCEAN  20640 non-null  bool   
dtypes: bool(4), float64(9)
memory u

In [46]:
df_lgb['ocean_proximity_NEAR BAY'].value_counts()

ocean_proximity_NEAR BAY
False    18350
True      2290
Name: count, dtype: int64

In [47]:
df_lgb.rename(columns={'ocean_proximity_NEAR BAY':'ocean_proximity_NEAR_BAY'},inplace=True)
df_lgb.rename(columns={'ocean_proximity_NEAR OCEAN':'ocean_proximity_NEAR_OCEAN'},inplace=True)

In [48]:
df_lgb.columns

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'ocean_proximity_INLAND',
       'ocean_proximity_ISLAND', 'ocean_proximity_NEAR_BAY',
       'ocean_proximity_NEAR_OCEAN'],
      dtype='object')

In [50]:
bool_cols = df_lgb.select_dtypes(include='bool').columns
df_lgb[bool_cols] = df_lgb[bool_cols].astype(int)

In [51]:
df_lgb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   longitude                   20640 non-null  float64
 1   latitude                    20640 non-null  float64
 2   housing_median_age          20640 non-null  float64
 3   total_rooms                 20640 non-null  float64
 4   total_bedrooms              20433 non-null  float64
 5   population                  20640 non-null  float64
 6   households                  20640 non-null  float64
 7   median_income               20640 non-null  float64
 8   median_house_value          20640 non-null  float64
 9   ocean_proximity_INLAND      20640 non-null  int64  
 10  ocean_proximity_ISLAND      20640 non-null  int64  
 11  ocean_proximity_NEAR_BAY    20640 non-null  int64  
 12  ocean_proximity_NEAR_OCEAN  20640 non-null  int64  
dtypes: float64(9), int64(4)
memory 

In [54]:
df_lgb['total_bedrooms']=df_lgb['total_bedrooms'].fillna(df_lgb['total_bedrooms'].median())

In [55]:
df_lgb.apply(skew).sort_values(ascending=False)

ocean_proximity_ISLAND        64.226165
population                     4.935500
total_rooms                    4.147042
total_bedrooms                 3.480888
households                     3.410190
ocean_proximity_NEAR_BAY       2.477478
ocean_proximity_NEAR_OCEAN     2.216540
median_income                  1.646537
median_house_value             0.977692
ocean_proximity_INLAND         0.784625
latitude                       0.465919
housing_median_age             0.060326
longitude                     -0.297780
dtype: float64

In [62]:
for col in X_train.columns:
    count=(X_train[col]<=0).sum()
    print(f'{col} {count}')

longitude 13698
latitude 0
housing_median_age 0
total_rooms 0
total_bedrooms 0
population 0
households 0
median_income 0
ocean_proximity_INLAND 9119
ocean_proximity_ISLAND 13695
ocean_proximity_NEAR BAY 12273
ocean_proximity_NEAR OCEAN 12002


In [63]:
pt_X=PowerTransformer(method='yeo-johnson')

In [64]:
X_lgb=df_lgb.drop('median_house_value',axis=1)
y_lgb=df_lgb['median_house_value']
X_lgb_train,X_lgb_test,y_lgb_train,y_lgb_test=train_test_split(X_lgb,y_lgb,test_size=0.25,random_state=15)

In [68]:
X_lgb_train_transformed=pt_X.fit_transform(X_lgb_train)
X_lgb_test_transformed=pt_X.transform(X_lgb_test)
col_name=pt_X.get_feature_names_out()


In [69]:
X_lgb_train_transformed_df=pd.DataFrame(X_lgb_train_transformed,columns=col_name)
X_lgb_test_transformed_df=pd.DataFrame(X_lgb_test_transformed,columns=col_name)
y_lgb_train_transformed,lambda_y=boxcox(y_lgb_train)

In [72]:
lgbmodel=LGBMRegressor(verbose=1)

In [73]:
lgbmodel.fit(X_lgb_train_transformed_df,y_lgb_train_transformed)

C:\Users\dolap\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Sistem belirtilen dosyayı bulamıyor
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\dolap\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\dolap\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dolap\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001230 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1799
[LightGBM] [Info] Number of data points in the train set: 15480, number of used features: 11
[LightGBM] [Info] Start training from score 27.943747


LGBMRegressor(verbose=1)

In [76]:
y_pred_transformed=lgbmodel.predict(X_lgb_test_transformed_df)
y_pred_lgb=inverse_boxcox(y_pred_transformed,lambda_y)

In [81]:
y_pred_lgb

array([181013.78248814, 112545.49485326, 125866.31215617, ...,
       331264.53069878, 348710.14732005, 123231.48675677])

In [82]:
print('mse:',mean_squared_error(y_lgb_test,y_pred_lgb))
print('msa:',mean_absolute_error(y_lgb_test,y_pred_lgb))
print('rmse:',np.sqrt(mean_squared_error(y_lgb_test,y_pred_lgb)))
print('score:',r2_score(y_lgb_test,y_pred_lgb))

mse: 2350909726.216218
msa: 31766.185939009723
rmse: 48486.180775724315
score: 0.8232636265465614


In [84]:
model=LGBMRegressor(subsample=0.6,reg_alpha=0.5,reg_lambda=1.0,
                        num_leaves=50,n_estimators=100,min_child_samples=30,
                        max_depth=-1,learning_rate=0.1,colsample_bytree=0.8)

In [85]:
params={'num_leaves':[31,50,70],'max_depth':[-1,5,10],
        'learning_rate':[0.01,0.05,0.1],'n_estimators':[100,300,1000],
        'min_child_samples':[10,20,30],'subsample':[0.6,0.8,1.0],
        'colsample_bytree':[0.6,0.8,1.0],'reg_alpha':[0,0.5,1.0],'reg_lambda':[0,0.5,1.0]
       }

In [86]:
random=RandomizedSearchCV(estimator=LGBMRegressor(),param_distributions=params,cv=5,n_jobs=-1)

In [87]:
random.fit(X_lgb_train_transformed_df,y_lgb_train_transformed)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000935 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1799
[LightGBM] [Info] Number of data points in the train set: 15480, number of used features: 11
[LightGBM] [Info] Start training from score 27.943747


RandomizedSearchCV(cv=5, estimator=LGBMRegressor(), n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.6, 0.8, 1.0],
                                        'learning_rate': [0.01, 0.05, 0.1],
                                        'max_depth': [-1, 5, 10],
                                        'min_child_samples': [10, 20, 30],
                                        'n_estimators': [100, 300, 1000],
                                        'num_leaves': [31, 50, 70],
                                        'reg_alpha': [0, 0.5, 1.0],
                                        'reg_lambda': [0, 0.5, 1.0],
                                        'subsample': [0.6, 0.8, 1.0]})

In [89]:
y_pred_transformed_tuning=random.predict(X_lgb_test_transformed_df)

In [90]:
y_pred_lgb_tuning=inverse_boxcox(y_pred_transformed_tuning,lambda_y)

In [91]:
print('LGBM Tuned ')
print('mse:',mean_squared_error(y_lgb_test,y_pred_lgb_tuning))
print('msa:',mean_absolute_error(y_lgb_test,y_pred_lgb_tuning))
print('rmse:',np.sqrt(mean_squared_error(y_lgb_test,y_pred_lgb_tuning)))
print('score:',r2_score(y_lgb_test,y_pred_lgb_tuning))

mse: 2116190027.6314862
msa: 29669.479818587024
rmse: 46002.06547136211
score: 0.8409093523025721


In [ ]:
print('LGBM NO Tuning')
print('mse:',mean_squared_error(y_lgb_test,y_pred_lgb))
print('msa:',mean_absolute_error(y_lgb_test,y_pred_lgb))
print('rmse:',np.sqrt(mean_squared_error(y_lgb_test,y_pred_lgb)))
print('score:',r2_score(y_lgb_test,y_pred_lgb))